In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
.appName('MySparkApp3')\
.getOrCreate()



Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/09 07:11:46 WARN Utils: Your hostname, MacBook-Pro-6.local, resolves to a loopback address: 127.0.0.1; using 100.100.165.223 instead (on interface en0)
26/09/09 07:11:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/09 07:11:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
listings = spark.read.csv("/Users/kamari/Documents/project_info/air_bnb/listings.csv.gz", 
    header=True,
    inferSchema=True,
    sep=",", 
    quote='"',
    escape='"', 
    multiLine=True,
    mode="PERMISSIVE" 
)

In [3]:
reviews = spark.read.csv("/Users/kamari/Documents/project_info/air_bnb/reviews.csv.gz", 
    header=True,
    inferSchema=True,
    sep=",", 
    quote='"',
    escape='"', 
    multiLine=True,
    mode="PERMISSIVE" 
)


In [ ]:
#UDF Practice #1 — Label Review Counts
from pyspark.sql import functions as f
from pyspark.sql.types import *

def review_count(num):
    if num < 20:
        return 'Low'
    else:
        return 'High'

my_udf = f.udf(review_count, StringType())

listings\
    .withColumn('review_level',
    my_udf(listings.number_of_reviews))\
    .select(f.col('review_level'))\
    .show()


+------------+
|review_level|
+------------+
|         Low|
|        High|
|        High|
|        High|
|        High|
|        High|
|        High|
|        High|
|        High|
|        High|
|        High|
|        High|
|        High|
|        High|
|        High|
|        High|
|        High|
|        High|
|        High|
|        High|
+------------+
only showing top 20 rows


In [13]:
#UDF Practice #2 — Listing Size: Create a UDF that labels each listing based on how many guests it accommodates:

def listing_sizes(num):
    if num <=2:
        return 'Small'
    elif 3 <= num <= 5:
        return 'Medium'
    else:
        return 'Large'
    
my_udf = f.udf(listing_sizes, StringType())

listings\
    .withColumn("listing_size",
    my_udf(listings.accommodates))\
    .select(f.col('listing_size'))\
    .show()
    

+------------+
|listing_size|
+------------+
|      Medium|
|      Medium|
|       Small|
|       Small|
|       Small|
|       Small|
|       Large|
|       Small|
|       Small|
|       Large|
|       Small|
|       Large|
|       Small|
|       Small|
|       Small|
|       Small|
|       Large|
|       Small|
|       Large|
|       Large|
+------------+
only showing top 20 rows


In [16]:
#UDF Practice #3 — Price Category: Create a UDF that labels each listing based on its numeric price:

from pyspark.sql.functions import regexp_replace

listings = listings.withColumn('price_numeric', regexp_replace('price', '[$,]', '').cast('float'))

def price_check(num):
    if num < 100:
        return "Budget"
    elif 100 <= num <= 249:
        return "Standard"
    else:
        return 'Expensive'
    
price_udf = f.udf(price_check, StringType())


listings\
    .withColumn('price_category', price_udf(listings.price_numeric))\
    .select(listings.price_numeric , f.col('price_category'))\
    .show()

+-------------+--------------+
|price_numeric|price_category|
+-------------+--------------+
|        80.43|        Budget|
|        211.0|      Standard|
|         97.0|        Budget|
|        160.0|      Standard|
|         86.0|        Budget|
|         58.0|        Budget|
|       277.31|     Expensive|
|        62.87|        Budget|
|         64.0|        Budget|
|        313.0|     Expensive|
|        51.89|        Budget|
|        394.0|     Expensive|
|         68.5|        Budget|
|         92.0|        Budget|
|        60.67|        Budget|
|        52.43|        Budget|
|        411.5|     Expensive|
|         92.0|        Budget|
|        388.0|     Expensive|
|       246.49|      Standard|
+-------------+--------------+
only showing top 20 rows
